In [ ]:
# -*- coding: utf-8 -*-
import sys
import os
import gc
import json
import h5py
import pandas as pd
import numpy as np
from tqdm import tqdm
from scipy.stats import gaussian_kde

# Library untuk Visualisasi Grafis Jurnal
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================================================================
# PERTAHANAN MURNI CPU & ANTI-LEAK (0-BYTE MEMORY LEAK ASSURANCE)
# ==============================================================================
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ["CUDA_VISIBLE_DEVICES"] = "-1" 

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU') 
from tensorflow import keras
from tensorflow.keras import backend as K
# ==============================================================================

BASE_REP = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo'
if BASE_REP not in sys.path:
    sys.path.append(BASE_REP)

from Library import utils, dataset

def predict_and_evaluate_1c_batch(model, batch_waves, kde_noise, kde_le):
    """
    Isolasi fungsional untuk komputasi batch guna mengembalikan 
    prediksi biner secara bersih tanpa kebocoran memori RAM.
    """
    num_points = 700
    batch_input = np.array(batch_waves, dtype=np.float32).reshape(-1, num_points, 1)
    
    with tf.device('/CPU:0'):
        embeddings = model.predict_on_batch(batch_input)
    
    laten_space = embeddings.T 
    like_noise = kde_noise.pdf(laten_space)
    like_le = kde_le.pdf(laten_space)
    
    likelihoods = np.vstack([like_noise, like_le])
    preds = np.argmax(likelihoods, axis=0)
    
    return preds

if __name__ == "__main__":
    # --- PATH SUMBER DATA UTAMA ---
    CSV_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.csv'
    HDF5_PATH = '/Volumes/Extreme SSD/stream_stead/data_stead/merge.hdf5'
    ZHI_GENG_JSON = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Benchmark_ STEAD 3C_ test n15275 r100/STEAD data, test n15275 r100.json'
    
    # --- PATH OUTPUT GRAFIS HASIL UNIFIKASI ---
    DIR_OUTPUT_GRAFIS = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/output_grafis_100k_stead'
    os.makedirs(DIR_OUTPUT_GRAFIS, exist_ok=True)
    
    MODEL_PATH = os.path.join(BASE_REP, "Pre-trained model/MCU-Quake 5-20")
    EMB_DIR = os.path.join(BASE_REP, "Typical embedding/Embedding_data train 3C, UUSS n11275 std15, 30120909")

    # --------------------------------------------------------------------------
    # FASE 0: PEMULIHAN FUNGSI PDF 1D BERFUNGSI (Z-AXIS ONLY)
    # --------------------------------------------------------------------------
    print("[INFO] Memuat Model dan Membangun Kurva PDF 1D Murni Zhi Geng (Z-Axis)...")
    with tf.device('/CPU:0'):
        embedding_model = keras.models.load_model(filepath=MODEL_PATH, compile=False)
    
    embedding_Z = dataset.load_embedding_data(EMB_DIR, "Embedding data, Z.json")
    
    available_keys = list(embedding_Z.keys())
    k_noise = next((k for k in available_keys if k.lower() in ['noise', 'no']), available_keys[0])
    k_le = next((k for k in available_keys if k.lower() in ['le', 'earthquake', 'eq']), available_keys[1] if len(available_keys)>1 else available_keys[0])

    kde_noise = gaussian_kde(np.array(embedding_Z[k_noise]).T)
    kde_le = gaussian_kde(np.array(embedding_Z[k_le]).T)
    
    del embedding_Z
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 1 & 2: ANTI-LEAKAGE FILTER & DATA BALANCING (100K STEAD)
    # --------------------------------------------------------------------------
    print("\n[INFO] Mengekstraksi daftar hitam anti-leakage dari JSON...")
    with open(ZHI_GENG_JSON, 'r') as f:
        zhi_geng_data = json.load(f)
    zhi_geng_traces = set(zhi_geng_data.keys()) if isinstance(zhi_geng_data, dict) else set(zhi_geng_data)
        
    df_raw = pd.read_csv(CSV_PATH, low_memory=False)
    df_filtered = df_raw[df_raw['trace_category'].isin(['earthquake_local', 'noise'])]
    df_unseen = df_filtered[~df_filtered['trace_name'].isin(zhi_geng_traces)]
    
    df_eq = df_unseen[df_unseen['trace_category'] == 'earthquake_local']
    df_noise = df_unseen[df_unseen['trace_category'] == 'noise']
    
    n_samples = 50000 
    df_eq_sample = df_eq.sample(n=min(n_samples, len(df_eq)), random_state=42)
    df_noise_sample = df_noise.sample(n=min(n_samples, len(df_noise)), random_state=42)
    
    df_final = pd.concat([df_eq_sample, df_noise_sample]).sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"[INFO] Total target uji STEAD 1C Murni: {len(df_final):,}")
    
    trace_names = df_final['trace_name'].to_numpy()
    trace_categories = df_final['trace_category'].to_numpy()
    p_arrivals = df_final['p_arrival_sample'].fillna(0).to_numpy().astype(np.int32)
    
    del df_raw, df_filtered, df_unseen, df_eq, df_noise, df_eq_sample, df_noise_sample, df_final
    gc.collect()

    # --------------------------------------------------------------------------
    # FASE 3: INFERENSI TERMINAL EXACT REPLICATION (RESTORED ATURAN RIGID)
    # --------------------------------------------------------------------------
    num_points = 700          # Durasi jendela input 7 detik
    norm_points = 900         # Durasi normalisasi 9 detik
    BUFFER_SIZE = 128
    buffer_waves = []
    
    # Kontainer array global kompak untuk pelacakan metrik akhir dan grafis
    all_true_labels = []
    all_pred_labels = []

    print(f"\n[INFO] Memulai eksekusi replikasi 1C murni Zhi Geng pada data STEAD...")
    with h5py.File(HDF5_PATH, 'r') as f_h5:
        data_group = f_h5['data']
        
        for idx in tqdm(range(len(trace_names)), desc="STEAD Restored 1C 100K"):
            try:
                trace_id = trace_names[idx]
                category = trace_categories[idx]
                
                if trace_id not in data_group:
                    continue
                    
                # [ATURAN 1] Ekstraksi komponen Vertikal murni (indeks 2) tanpa filter frekuensi
                raw_wave = data_group[trace_id][:, 2]
                
                # [ATURAN 2] Penghilangan tren (Detrending) secara EKSKLUSIF pada gelombang mentah kontinu
                detrended_wave = raw_wave - np.mean(raw_wave)
                
                if category == 'earthquake_local':
                    p_arrival = p_arrivals[idx]
                    
                    # [ATURAN 3] Pemotongan jendela masukan dimulai PERSIS pada kedatangan gelombang-P
                    start_idx = int(p_arrival)
                    end_idx = start_idx + num_points
                    norm_end_idx = start_idx + norm_points
                    
                    if norm_end_idx > len(detrended_wave) or start_idx < 0: 
                        continue
                        
                    z_component = detrended_wave[start_idx:end_idx]
                    
                    # [ATURAN 4] Cari nilai maksimum spesifik di dalam jendela 9 detik setelah P-arrival
                    nine_second_window = detrended_wave[start_idx:norm_end_idx]
                    norm_val = np.max(np.abs(nine_second_window))
                    true_label = 1
                else:
                    # Untuk kelas kebisingan (ambient noise): potongan awal rekaman lingkungan mendahului gelombang P
                    z_component = detrended_wave[:num_points]
                    norm_val = np.max(np.abs(detrended_wave[:norm_points]))
                    true_label = 0
                
                if norm_val == 0:
                    norm_val = 1e-8
                    
                z_component /= norm_val
                
                # [ATURAN 5] Resampling 100 Hz (Naturally 100Hz pada STEAD, langsung bypass masuk antrean)
                buffer_waves.append(z_component)
                all_true_labels.append(true_label)
                
                if len(buffer_waves) == BUFFER_SIZE:
                    b_preds = predict_and_evaluate_1c_batch(
                        embedding_model, buffer_waves, kde_noise, kde_le
                    )
                    all_pred_labels.extend(b_preds)
                    buffer_waves.clear()
                    
                if (idx + 1) % 10000 == 0:
                    gc.collect() 
                    K.clear_session()
                    
            except Exception:
                continue

        # Sisa Eksekusi Akhir di dalam Buffer
        if len(buffer_waves) > 0:
            b_preds = predict_and_evaluate_1c_batch(
                embedding_model, buffer_waves, kde_noise, kde_le
            )
            all_pred_labels.extend(b_preds)

    # Konversi hasil kontainer akhir menjadi array NumPy murni
    y_true = np.array(all_true_labels, dtype=np.int32)
    y_pred = np.array(all_pred_labels, dtype=np.int32)

    # Membangun struktur data elemen Confusion Matrix riil
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    # Kalkulasi Akurasi & Metrik Marginal Numerik
    total_data = TP + TN + FP + FN
    akurasi = (TP + TN) / total_data if total_data > 0 else 0
    tpr_recall = TP / (TP + FN) if (TP + FN) > 0 else 0          # TPR Gempa
    tnr_spesifisitas = TN / (TN + FP) if (TN + FP) > 0 else 0    # TNR / TPR Noise
    ppv_presisi = TP / (TP + FP) if (TP + FP) > 0 else 0         # PPV Gempa
    f1_score = 2 * (ppv_presisi * tpr_recall) / (ppv_presisi + tpr_recall) if (ppv_presisi + tpr_recall) > 0 else 0

    tpr_no = tnr_spesifisitas * 100
    tpr_le = tpr_recall * 100
    ppv_no = (TN / (TN + FN) * 100) if (TN + FN) > 0 else 0.0
    ppv_le = ppv_presisi * 100

    # ==========================================================================
    # FASE 4: OTOMASI EKSEKUSI BLOK OUTPUT GRAFIS (ZHI GENG STYLE)
    # ==========================================================================
    print(f"\n[INFO] Menghasilkan grafik visualisasi ilmiah ke path: {DIR_OUTPUT_GRAFIS}")
    
    # --- GRAFIK 1: CONFUSION MATRIX GAYA KLASIK ORIGINAL ZHI GENG ---
    cm_matrix = np.array([[TN, FP], [FN, TP]])
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    cax = ax.matshow(cm_matrix, cmap=plt.cm.Blues, vmin=0, vmax=np.max(cm_matrix))

    threshold = np.max(cm_matrix) / 2.
    for i in range(2):
        for j in range(2):
            color = "white" if cm_matrix[i, j] > threshold else "#08306b"
            ax.text(j, i, format(cm_matrix[i, j], 'd'), ha="center", va="center", color=color, fontsize=15, weight='bold')

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(1.5, -0.5)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['NO.', 'LE.'], fontsize=13, weight='bold') 
    ax.set_yticklabels(['NO.', 'LE.'], fontsize=13, weight='bold')
    ax.yaxis.set_ticks_position('left')
    ax.xaxis.set_ticks_position('top')
    
    ax.set_ylabel('True Label (Ground Truth)', fontsize=13, weight='bold', labelpad=10)
    ax.set_xlabel('Predicted Label (Model Inferens)', fontsize=13, weight='bold', labelpad=12)
    ax.xaxis.set_label_position('top') 
    
    ax.text(-0.5, -0.8, 'Baseline MCU-Quake, STEAD Unseen, 1C', ha='left', va='center', fontsize=13, weight='bold', color='#c92a2a', clip_on=False)

    # Proyeksi Teks Kanan Kotak (TPR Marginal)
    jarak_x_tpr = 1.7 
    ax.text(jarak_x_tpr, -0.4, 'TPR:', ha='center', va='center', fontsize=12, weight='bold', color='black', clip_on=False)
    ax.text(jarak_x_tpr, 0, f"{tpr_no:.2f}%", ha='center', va='center', fontsize=12, weight='bold', color='#c92a2a', clip_on=False)
    ax.text(jarak_x_tpr, 1, f"{tpr_le:.2f}%", ha='center', va='center', fontsize=12, weight='bold', color='blue', clip_on=False)

    # Proyeksi Teks Bawah Kotak (PPV Marginal)
    jarak_y_ppv = 1.7 
    ax.text(-0.4, jarak_y_ppv, 'PPV:', ha='center', va='center', fontsize=12, weight='bold', color='black', clip_on=False)
    ax.text(0, jarak_y_ppv, f"{ppv_no:.2f}%", ha='center', va='center', fontsize=12, weight='bold', color='black', clip_on=False)
    ax.text(1, jarak_y_ppv, f"{ppv_le:.2f}%", ha='center', va='center', fontsize=12, weight='bold', color='black', clip_on=False)

    for edge, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1.8)
    ax.grid(False)
    plt.tight_layout()
    
    out_path_cm = os.path.join(DIR_OUTPUT_GRAFIS, "1C_STEAD_ZhiGeng_Style_CM.png")
    plt.savefig(out_path_cm, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[SUKSES] Grafik 1: Confusion Matrix tersimpan -> {out_path_cm}")

    # --- GRAFIK 2: DIAGRAM BATANG METRIK KOMPREHENSIF STANDARD JURNAL ---
    metrics_names = ['Akurasi', 'TPR\n(Recall)', 'TNR\n(Spesifisitas)', 'PPV\n(Presisi)', 'F1-Score']
    metrics_scores = [akurasi*100, tpr_recall*100, tnr_spesifisitas*100, ppv_presisi*100, f1_score*100]

    plt.figure(figsize=(9, 5.5))
    ax_bar = sns.barplot(x=metrics_names, y=metrics_scores, hue=metrics_names, palette="mako", legend=False)
    
    plt.title("Metrik Evaluasi - Exact Replication MCU-Quake (1C, Data STEAD Unseen)", fontsize=12, weight='bold', pad=15)
    plt.ylabel("Persentase (%)", fontsize=11, weight='bold')
    plt.ylim(0, 115)  
    
    for i, p in enumerate(ax_bar.patches):
        ax_bar.annotate(f'{metrics_scores[i]:.2f}%', 
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', fontsize=11, fontweight='bold', color='black', 
                    xytext=(0, 8), textcoords='offset points')

    ax_bar.spines['left'].set_linewidth(1.5)
    ax_bar.spines['bottom'].set_linewidth(1.5)
    sns.despine() 
    plt.tight_layout()
    
    out_path_bar = os.path.join(DIR_OUTPUT_GRAFIS, "1C_STEAD_Metrics_BarChart.png")
    plt.savefig(out_path_bar, dpi=300)
    plt.close()
    print(f"[SUKSES] Grafik 2: Diagram Batang Metrik tersimpan -> {out_path_bar}")

    # ==========================================================================
    # TERMINAL OUTPUT REPORTING
    # ==========================================================================
    print("\n=======================================================")
    print(" HASIL RESTORED EXACT REPLICATION ZHI GENG (STEAD 100K 1C)")
    print("=======================================================")
    print(f"Total Data Terproses  : {total_data:,} sampel")
    print(f"True Positives (TP)   : {TP:,}")
    print(f"True Negatives (TN)   : {TN:,}")
    print(f"False Positives (FP)  : {FP:,}")
    print(f"False Negatives (FN)  : {FN:,}")
    print("-------------------------------------------------------")
    print(f"Akurasi Global        : {akurasi:.4f} ({(akurasi*100):.2f}%)")
    print(f"Recall (TPR)          : {recall_tpr:.4f} ({(recall_tpr*100):.2f}%)")
    print(f"Spesifisitas (TNR)    : {tnr_spesifisitas:.4f} ({(tnr_spesifisitas*100):.2f}%)")
    print(f"Presisi (PPV)         : {ppv_presisi:.4f} ({(ppv_presisi*100):.2f}%)")
    print(f"F1-Score              : {f1_score:.4f} ({(f1_score*100):.2f}%)")
    print("=======================================================")
    print("\n[INFO] Seluruh proses inferensi dan visualisasi SELESAI. NYALAKAN!")

[INFO] Memuat Model dan Membangun Kurva PDF 1D Murni Zhi Geng (Z-Axis)...

[INFO] Mengekstraksi daftar hitam anti-leakage dari JSON...
[INFO] Total target uji STEAD 1C Murni: 100,000

[INFO] Memulai eksekusi replikasi 1C murni Zhi Geng pada data STEAD...


STEAD Restored 1C 100K: 100%|██████████| 100000/100000 [23:16<00:00, 71.63it/s]



[INFO] Menghasilkan grafik visualisasi ilmiah ke path: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/output_grafis_100k_stead
[SUKSES] Grafik Confusion Matrix tersimpan -> /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/output_grafis_100k_stead/1c_restored_confusion_matrix.png
[SUKSES] Grafik Kurva Performa (ROC & PR) tersimpan -> /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/output_grafis_100k_stead/1c_restored_performance_curves.png

 HASIL RESTORED EXACT REPLICATION ZHI GENG (STEAD 100K 1C)
Total Data Terproses  : 100,000 sampel
True Positives (TP)   : 49,345
True Negatives (TN)   : 249
False Positives (FP)  : 49,751
False Negatives (FN)  : 655
-------------------------------------------------------
Akurasi Global        : 0.4959 (49.59%)
Recall (TPR)          : 0.9869 (98.69%)
Spesifisitas (